<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep — Audio Stem Separator Demo

Welcome to the **vsep** interactive demo! Separate audio into stems (vocals, drums, bass, instruments) using state-of-the-art AI models.

## Features
- **Fast Processing** — GPU-accelerated inference on Colab
- **Multiple Models** — 21 curated models across 6 categories
- **YouTube Support** — Paste a URL or upload a file
- **Leaderboard** — Browse top models ranked by SDR quality
- **Easy to Use** — Upload, pick a model, separate, download

See the [full documentation](https://github.com/BF667-IDLE/vsep/tree/main/docs) for more details.

---

## 1. Setup

Install vsep, yt-dlp, and dependencies. This takes about 2–3 minutes.

In [ ]:
#@title Install vsep
#@markdown Run this cell to install vsep (first time only)

# Clone the repository
!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep

# Install dependencies
!pip install -q -r /content/vsep/requirements.txt

# Install ffmpeg (required by pydub for audio I/O)
!apt-get -qq install ffmpeg > /dev/null 2>&1

# Install yt-dlp (for YouTube downloads)
!pip install -q yt-dlp

import os
os.chdir('/content/vsep')

import torch
if torch.cuda.is_available():
    print(f"NVIDIA GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, will use CPU (slower)")
    print("Tip: Go to Runtime > Change runtime type > Hardware accelerator > GPU")

# Verify yt-dlp
import yt_dlp
print(f"yt-dlp {yt_dlp.version.__version__} ready")

## 2. Get Your Audio

Choose one method: **upload a file** from your device, or **paste a YouTube URL** to download directly.

In [ ]:
#@title Method A — Upload Audio File
#@markdown Upload your audio file (MP3, WAV, FLAC, OGG, M4A)

from google.colab import files

print("Please upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f"\nUploaded: {audio_file}")
else:
    print("No file uploaded.")

In [ ]:
#@title Method B — Download from YouTube
#@markdown Paste a YouTube / SoundCloud / Bandcamp URL to download the audio

import yt_dlp, os

youtube_url = "" #@param {type:"string"}

if youtube_url.strip():
    os.makedirs("/content/vsep/audio_input", exist_ok=True)
    
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': '/content/vsep/audio_input/%(title).80s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '0',
        }],
        'quiet': False,
        'no_warnings': False,
    }

    try:
        print(f"Downloading audio from:\n  {youtube_url}\n")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url.strip()])

        # Find the downloaded file
        audio_files = sorted(
            f for f in os.listdir("/content/vsep/audio_input")
            if f.endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a"))
        )
        if audio_files:
            audio_file = f"/content/vsep/audio_input/{audio_files[-1]}"
            print(f"\nDownloaded: {audio_file}")
        else:
            print("Download completed but no audio file found.")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Tip: Make sure the URL is correct and the video is publicly available.")
else:
    print("No URL provided. Use Method A to upload a file instead.")

## 3. Model Leaderboard

Browse the top models ranked by quality. Select a category, then pick a model.

In [ ]:
#@title Model Leaderboard#@markdown Browse top models by category. Use the dropdown to filter.# Model catalog: display_name -> actual model filename# The Select Model cell uses the filename directly.MODEL_CATALOG = {    # --- Vocals (best vocal isolation) ---    "model_bs_roformer_ep_317_sdr_12.9755.ckpt": {        "display_name": "BS Roformer EP 317 SDR 12.98",        "arch": "Roformer",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 12.98,    },    "Mel-Roformer-Viperx-1053.ckpt": {        "display_name": "Mel Roformer Viperx SDR 12.61",        "arch": "Roformer",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 12.61,    },    "BS-Roformer-Viperx-1297.ckpt": {        "display_name": "BS Roformer Viperx 1297",        "arch": "Roformer",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 12.50,    },    "MDX23C-8KFFT-InstVoc_HQ.ckpt": {        "display_name": "MDX23C InstVoc HQ SDR 11.95",        "arch": "MDXC",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 11.95,    },    "UVR-MDX-NET-Inst_HQ_2.onnx": {        "display_name": "MDX Net Inst HQ 2 SDR 10.93",        "arch": "MDX-Net",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 10.93,    },    "UVR-MDX-NET-Inst_1.onnx": {        "display_name": "MDX Net Inst 1 SDR 10.65",        "arch": "MDX-Net",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 10.65,    },    "UVR-MDX-NET-Inst_HQ_4.onnx": {        "display_name": "MDX Net Inst HQ 4 SDR 10.49",        "arch": "MDX-Net",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 10.49,    },    "Kim_Vocal_2.onnx": {        "display_name": "Kim Vocal 2",        "arch": "MDX-Net",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 10.21,    },    "UVR-MDX-NET-KARA_2.onnx": {        "display_name": "MDX Net Karaoke 2",        "arch": "MDX-Net",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 9.50,    },    "5_HP-UVR.pth": {        "display_name": "VR 5 HP",        "arch": "VR",        "category": "Vocals",        "stems": "vocals, instrumental",        "sdr": 8.22,    },    # --- Multi-Stem (4+ stems) ---    "ht-demucs_ft.yaml": {        "display_name": "Demucs v4 Fine Tuned",        "arch": "Demucs",        "category": "Multi-Stem",        "stems": "vocals, drums, bass, other",        "sdr": 11.27,    },    "htdemucs_6s.yaml": {        "display_name": "Demucs v4 6 Stem",        "arch": "Demucs",        "category": "Multi-Stem",        "stems": "vocals, drums, bass, other, guitar, piano",        "sdr": 10.80,    },    # --- Reverb / Echo Removal ---    "MDX23C-8KFFT-Reverb_HQ.ckpt": {        "display_name": "Reverb HQ SDR 11.20",        "arch": "MDXC",        "category": "Reverb/Echo Removal",        "stems": "vocals, no reverb",        "sdr": 11.20,    },    "MDX23C-8KFFT-Echo_HQ.ckpt": {        "display_name": "Echo HQ SDR 11.10",        "arch": "MDXC",        "category": "Reverb/Echo Removal",        "stems": "vocals, no echo",        "sdr": 11.10,    },    # --- De-Esser ---    "MDX23C-8KFFT-DeEsser.ckpt": {        "display_name": "De-Esser MDX SDR 10.90",        "arch": "MDXC",        "category": "De-Esser",        "stems": "vocals, no sibilance",        "sdr": 10.90,    },    # --- Stem Isolation ---    "MDX23C-8KFFT-Guitar.ckpt": {        "display_name": "Guitar Isolation SDR 10.80",        "arch": "MDXC",        "category": "Stem Isolation",        "stems": "guitar, no guitar",        "sdr": 10.80,    },    "MDX23C-8KFFT-Drum.ckpt": {        "display_name": "Drum Isolation SDR 10.50",        "arch": "MDXC",        "category": "Stem Isolation",        "stems": "drums, no drums",        "sdr": 10.50,    },    "MDX23C-8KFFT-Bass.ckpt": {        "display_name": "Bass Isolation SDR 10.10",        "arch": "MDXC",        "category": "Stem Isolation",        "stems": "bass, no bass",        "sdr": 10.10,    },    "MDX23C-8KFFT-Piano.ckpt": {        "display_name": "Piano Isolation SDR 9.90",        "arch": "MDXC",        "category": "Stem Isolation",        "stems": "piano, no piano",        "sdr": 9.90,    },    # --- Instrumental / Karaoke ---    "MDX23C-8KFFT-InstVoc.ckpt": {        "display_name": "MDX23C InstVoc SDR 10.75",        "arch": "MDXC",        "category": "Instrumental",        "stems": "vocals, instrumental",        "sdr": 10.75,    },}# Build categoriescategories = {}for filename, info in MODEL_CATALOG.items():    cat = info["category"]    if cat not in categories:        categories[cat] = []    categories[cat].append((filename, info))# Sort each category by SDRfor cat in categories:    categories[cat].sort(key=lambda x: x[1]["sdr"], reverse=True)category = "Vocals" #@param ["Vocals", "Multi-Stem", "Reverb/Echo Removal", "De-Esser", "Stem Isolation", "Instrumental"]models_in_cat = categories[category]print(f"\n{'#':<4} {'Display Name':<38} {'Filename':<45} {'Arch':<8} {'SDR':>6}")print("=" * 105)for i, (filename, info) in enumerate(models_in_cat, 1):    print(f"{i:<4} {info['display_name']:<38} {filename:<45} {info['arch']:<8} {info['sdr']:>6.2f}")print(f"\n{len(models_in_cat)} models in '{category}' | Total catalog: {len(MODEL_CATALOG)} models")

## 4. Choose Model

Pick a model from the list. The display name is shown alongside the actual filename.
You can also type any custom filename directly.

In [ ]:
#@title Select Model#@markdown Pick from the list or type any model filename. The value is the actual model filename.# Build dropdown: sorted by SDR, showing display_name — filenamesorted_models = sorted(    MODEL_CATALOG.items(),    key=lambda x: x[1]["sdr"],    reverse=True,)# Create human-readable labels: Display Name — filenamemodel_labels = [    f"{info['display_name']}  ↔  {filename}"    for filename, info in sorted_models]# The dropdown value IS the actual filenamemodel_filenames = [filename for filename, info in sorted_models]# Single parameter: shows display name + filename, value = filenameselected_model = "model_bs_roformer_ep_317_sdr_12.9755.ckpt" #@param ["model_bs_roformer_ep_317_sdr_12.9755.ckpt", "Mel-Roformer-Viperx-1053.ckpt", "BS-Roformer-Viperx-1297.ckpt", "MDX23C-8KFFT-InstVoc_HQ.ckpt", "UVR-MDX-NET-Inst_HQ_2.onnx", "MDX23C-8KFFT-Reverb_HQ.ckpt", "MDX23C-8KFFT-Echo_HQ.ckpt", "MDX23C-8KFFT-DeEsser.ckpt", "UVR-MDX-NET-Inst_1.onnx", "MDX23C-8KFFT-Guitar.ckpt", "MDX23C-8KFFT-Drum.ckpt", "MDX23C-8KFFT-InstVoc.ckpt", "MDX23C-8KFFT-Bass.ckpt", "UVR-MDX-NET-Inst_HQ_4.onnx", "MDX23C-8KFFT-Piano.ckpt", "Kim_Vocal_2.onnx", "UVR-MDX-NET-KARA_2.onnx", "ht-demucs_ft.yaml", "htdemucs_6s.yaml", "5_HP-UVR.pth"]# Display infoif selected_model in MODEL_CATALOG:    info = MODEL_CATALOG[selected_model]    print(f"Model:     {info['display_name']}")    print(f"Filename:  {selected_model}")    print(f"Arch:      {info['arch']}")    print(f"Category:  {info['category']}")    print(f"Stems:     {info['stems']}")    print(f"SDR:       {info['sdr']}")else:    print(f"Model:     {selected_model}")    print(f"(custom model — not in catalog)")

## 5. Separate Audio

Run the separation on your audio file.

In [ ]:
#@title Run Separation#@markdown Click to start the separation processfrom separator import Separatorimport config.variables as cfg# Optimize for Colabcfg.MAX_DOWNLOAD_WORKERS = 4cfg.DOWNLOAD_CHUNK_SIZE = 262144print("Starting separation process...")print(f"Input file: {audio_file}")print(f"Model: {selected_model}")print("\nThis may take 2-5 minutes depending on the model and audio length...\n")# Initialize separatorseparator = Separator(    model_file_dir="/content/vsep/models",    output_dir="/content/vsep/output",    output_format="WAV",    use_soundfile=True,)# Load modelseparator.load_model(model_filename=selected_model)# Run separationoutput_files = separator.separate(audio_file)print(f"\nSeparation complete!")print(f"Output files: {len(output_files)}")for f in output_files:    print(f"  - {f}")

## 6. Listen to Results

Play back the separated stems to hear the results.

In [ ]:
#@title Listen to Separated Stems#@markdown Browse and play the separated audio filesimport globfrom IPython.display import Audio, displayoutput_files = sorted(glob.glob("/content/vsep/output/*.*"))if output_files:    print(f"Found {len(output_files)} separated stems:\n")    for i, file_path in enumerate(output_files, 1):        filename = os.path.basename(file_path)        print(f"{i}. {filename}")        display(Audio(file_path))        print("-" * 50)else:    print("No output files found. Please run the separation first.")

## 7. Download Results

Download the separated stems to your computer.

In [ ]:
#@title Download Separated Stems#@markdown Download all separated audio filesfrom google.colab import filesimport globoutput_files = glob.glob("/content/vsep/output/*.*")if output_files:    print(f"Downloading {len(output_files)} files...")    for file_path in output_files:        files.download(file_path)        print(f"Downloaded: {os.path.basename(file_path)}")else:    print("No files to download. Please run separation first.")

In [ ]:
#@title Browse Full Model List (Optional)
#@markdown Run this to see all 100+ supported models from UVR

from separator import Separator

sep = Separator(info_only=True)
models = sep.get_simplified_model_list(filter_sort_by="vocals")

print(f"Showing top 20 vocal models (out of {len(models)} total)\n")
print(f"{'#':<4} {'Model':<55} {'Arch':<10} {'Stems'}")
print("-" * 100)
for i, (filename, info) in enumerate(list(models.items())[:20], 1):
    stems = ", ".join(info["Stems"])
    print(f"{i:<4} {filename:<55} {info['Type']:<10} {stems}")

## Additional Information

### Quick Recommendations

| What You Need | Model Filename | Why |
|:---------------|:---------------|:-----|
| Clean vocals | `model_bs_roformer_ep_317_sdr_12.9755.ckpt` | Highest vocal SDR (12.98) |
| All 4 stems | `ht-demucs_ft.yaml` | Vocals + drums + bass + other |
| 6 stems | `htdemucs_6s.yaml` | Adds guitar and piano |
| Karaoke track | `UVR-MDX-NET-Inst_HQ_2.onnx` | Clean instrumental |
| Remove reverb | `MDX23C-8KFFT-Reverb_HQ.ckpt` | Isolates dry vocals |
| Remove echo | `MDX23C-8KFFT-Echo_HQ.ckpt` | Removes room echo |
| Remove sibilance | `MDX23C-8KFFT-DeEsser.ckpt` | De-esses vocals |
| Isolate guitar | `MDX23C-8KFFT-Guitar.ckpt` | Extracts guitar |
| Isolate drums | `MDX23C-8KFFT-Drum.ckpt` | Extracts drums |
| Isolate bass | `MDX23C-8KFFT-Bass.ckpt` | Extracts bass |
| Isolate piano | `MDX23C-8KFFT-Piano.ckpt` | Extracts piano |

### Audio Sources

- **Method A (Upload):** Supports MP3, WAV, FLAC, OGG, M4A files
- **Method B (YouTube):** Supports YouTube, SoundCloud, Bandcamp, and 1000+ sites via yt-dlp
  - Paste the full URL (e.g. `https://www.youtube.com/watch?v=...`)
  - Audio is downloaded as high-quality WAV automatically

### Tips

1. **GPU Acceleration:** Make sure you're using a GPU runtime for faster processing
   - Go to `Runtime > Change runtime type > Hardware accelerator > GPU`
2. **Long Files:** For files longer than 10 minutes, enable chunking to avoid memory errors
3. **Quality vs Speed:** Larger Roformer models produce better quality but take longer
4. **Memory:** If you run out of memory, try a smaller model (MDX-Net) or shorter audio

### Troubleshooting

**"No GPU detected"**
- Go to Runtime > Change runtime type > GPU
- Restart the runtime if needed

**"Download failed"**
- The model download may have timed out — re-run the separation cell (downloads are cached)
- Increase timeout: `cfg.DOWNLOAD_TIMEOUT = 600`

**"yt-dlp error"**
- Make sure the URL is correct and the video is publicly available
- Try a different URL or upload the file directly instead

**"Out of memory"**
- Try a shorter audio file or use `chunk_duration=300` when creating the Separator
- Use a simpler model (MDX-Net instead of Roformer)

**"Module not found"**
- Make sure you ran the installation cell first
- Try restarting the runtime (Runtime > Restart runtime)

---

### Links

- **GitHub:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- **API Reference:** [docs/API-Reference.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/API-Reference.md)
- **Architecture:** [docs/Architecture.md](https://github.com/BF667-IDLE/vsep/blob/main/docs/Architecture.md)
- **Install Guide:** [INSTALL.md](https://github.com/BF667-IDLE/vsep/blob/main/INSTALL.md)
- **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)

Made with care using vsep